[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Estructura del informe y comunicación de resultados.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo IV: *Proyecto integrador*

En este *notebook* se presentará la estructura de un informe de hallazgos y cómo comunicar números: intervalos de confianza y tamaños del efecto en lenguaje llano, la gráfica final de un hallazgo y lo que no se debe afirmar.

## 📚 Referencias.

* Markdown en *Jupyter*, para redactar el informe dentro del *notebook*: [py311 · 01 Introducción a *Jupyter*](https://github.com/PythonistaMX/py311/blob/main/01_introduccion_a_jupyter.ipynb).
* Exportación: [documentación de `nbconvert`](https://nbconvert.readthedocs.io/en/latest/usage.html).
* Gráficas: [py311 · 31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, `stats` de *SciPy* y los módulos de `nbformat` y `nbconvert` para la exportación, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
import tempfile

import nbformat
from nbconvert import HTMLExporter
from scipy import stats

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará desde `penguins.csv` si no existe) y `mpg.csv`, y creará `litros_100km`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
autos = pd.read_csv(os.path.join(DATA, "mpg.csv")).assign(litros_100km=lambda d: 235.215 / d["mpg"])
pinguinos.shape, autos.shape

---
## 1. Repaso (10 min).

En la sesión 7 se recorrió el EDA completo de `mpg` y cada participante arrancó su proyecto con preguntas y auditoría. El análisis ya está hecho; lo que falta es lo que leerá quien tome la decisión. Un análisis correcto mal comunicado produce las mismas malas decisiones que un análisis incorrecto.

| Del *notebook* de análisis | Al informe |
|---|---|
| Todo lo que se probó, en el orden en que se probó | Solo lo que responde a las preguntas, en orden de importancia |
| Salidas crudas (`describe`, tablas de *SciPy*) | Números redondeados con sus unidades e incertidumbre |
| Decenas de gráficas exploratorias | Una gráfica final por hallazgo, con título que dice el hallazgo |
| Código | Texto; el código queda disponible en el repositorio |

---
## 2. Estructura del informe (25 min).

| Sección | Qué contiene | Extensión sugerida |
|---|---|---|
| **Resumen ejecutivo** | Pregunta, respuesta principal con su número clave e incertidumbre, y la principal limitación. Se escribe **al final** y se lee **primero** | Máximo 150 palabras |
| **Datos y método** | Fuente, población que representan, cómo se recolectaron, decisiones de calidad (con número de filas afectadas) y métodos usados | Media página |
| **Hallazgos** | Uno por pregunta: afirmación, gráfica final, estadístico con IC, tamaño del efecto e interpretación | Media página por hallazgo |
| **Limitaciones** | Sesgos de recolección, confusores no controlados, supuestos dudosos, lo que **no** se puede concluir | Un párrafo o una lista |
| **Recomendaciones** | Qué hacer con los resultados y qué datos harían falta para responder lo que quedó abierto | 2 a 4 viñetas |
| **Reproducibilidad** | Repositorio, versión de las bibliotecas, semilla, cómo ejecutar | 2 o 3 líneas |

**Principio de la pirámide invertida:** lo más importante va primero. Quien solo lea el resumen ejecutivo debe quedarse con la respuesta correcta y con su principal límite.

**Ejemplo:** el resumen ejecutivo del EDA de `mpg` de la sesión 7. La extensión importa: un resumen de 400 palabras deja de ser un resumen. La siguiente función cuenta las palabras para verificarlo.

* La siguiente celda definirá `contar_palabras` y verificará que el resumen ejecutivo de `mpg` no pase de 150 palabras.

In [ ]:
def contar_palabras(texto):
    return len(texto.split())


resumen_mpg = (
    "Se analizaron 398 modelos de automóviles vendidos en EE. UU. entre 1970 y 1982 para saber qué "
    "características se asocian con el consumo de combustible y si este difiere según el origen. El consumo "
    "se asocia sobre todo con el tamaño del auto: peso, motor, potencia y cilindros miden casi lo mismo, y el "
    "peso por sí solo se relaciona de forma casi lineal con los litros por cada 100 km (r = 0.89). Los modelos "
    "de EE. UU. consumen unos 4 a 5 L/100 km más que los europeos y japoneses, pero también pesan unas 1,000 lb "
    "más; entre autos ligeros de peso parecido, la diferencia baja a menos de 1 L/100 km. Principal limitación: "
    "cada fila es un modelo, no un auto vendido, y no hay autos pesados europeos o japoneses con los cuales "
    "comparar."
)
contar_palabras(resumen_mpg)

**Interpretación:** el resumen tiene 137 palabras. Observa su estructura: (1) qué se analizó y para qué, (2) la respuesta a la primera pregunta con su número clave, (3) la respuesta a la segunda **con el confusor incluido**, y (4) la limitación principal. No tiene valores *p*, nombres de pruebas ni código: esos detalles van en la sección de hallazgos.

**Error común que se evita:** un resumen que solo describe lo que se hizo ("se limpiaron los datos, se hicieron gráficas y pruebas") sin decir **qué se encontró**.

**💡 Ideas clave de la sección 2:**
* Seis secciones: resumen ejecutivo, datos y método, hallazgos, limitaciones, recomendaciones, reproducibilidad.
* El resumen ejecutivo responde, con números, en máximo 150 palabras, e incluye la principal limitación.
* Lo más importante va primero.

---
## 3. Comunicar números (20 min).

### 3.1 Un IC y un tamaño del efecto en lenguaje llano.

Un hallazgo bien comunicado responde tres preguntas: **¿cuánto?** (la estimación en unidades originales), **¿con qué seguridad?** (el IC) y **¿importa?** (el tamaño del efecto en términos que el lector entienda). Se usará la diferencia de masa entre machos y hembras Gentoo, ya calculada en las sesiones 5 y 6.

* La siguiente celda calculará la diferencia de masa media entre machos y hembras Gentoo con su IC *bootstrap* del 95% y la d de Cohen.

In [ ]:
gentoo = pinguinos[pinguinos["species"] == "Gentoo"]
machos = gentoo.loc[gentoo["sex"] == "Male", "body_mass_g"].to_numpy()
hembras = gentoo.loc[gentoo["sex"] == "Female", "body_mass_g"].to_numpy()


def diferencia_medias(a, b, axis=-1):
    return np.mean(a, axis=axis) - np.mean(b, axis=axis)


ic = stats.bootstrap((machos, hembras), diferencia_medias, n_resamples=9999, method="percentile",
                     random_state=401).confidence_interval
varianza_combinada = ((len(machos) - 1) * machos.var(ddof=1) + (len(hembras) - 1) * hembras.var(ddof=1)) / (len(machos) + len(hembras) - 2)
d = diferencia_medias(machos, hembras) / np.sqrt(varianza_combinada)
{"n": (len(machos), len(hembras)), "diferencia (g)": round(float(diferencia_medias(machos, hembras)), 1),
 "IC 95% (g)": (round(float(ic.low), 1), round(float(ic.high), 1)), "d de Cohen": round(float(d), 2)}

**Concepto: el tamaño del efecto en lenguaje común.** Una d de 2.70 no le dice nada a quien no estudió estadística. Una traducción intuitiva es la **probabilidad de superioridad**: si se elige al azar un macho y una hembra, ¿con qué probabilidad el macho pesa más? (los empates cuentan como medio caso). Si no hay diferencia entre los grupos, vale 0.5: es como lanzar una moneda.

Se puede obtener comparando todos los pares macho-hembra posibles o, de forma equivalente, con el estadístico *U* de Mann-Whitney de la sesión 6 dividido entre el número de pares.

* La siguiente celda calculará la probabilidad de superioridad de dos maneras: comparando todos los pares y con el estadístico *U* de Mann-Whitney.

In [ ]:
todos_los_pares = (machos[:, None] > hembras[None, :]).mean() + 0.5 * (machos[:, None] == hembras[None, :]).mean()
con_u = stats.mannwhitneyu(machos, hembras).statistic / (len(machos) * len(hembras))
round(float(todos_los_pares), 3), round(float(con_u), 3)

**Interpretación:** en 97.9% de los pares macho-hembra, el macho pesa más. Ahora el hallazgo se puede escribir para cualquier lector:

> "Los machos Gentoo pesan en promedio unos 800 g más que las hembras (entre 700 y 910 g, con 95% de confianza). La diferencia es muy grande: al comparar un macho y una hembra elegidos al azar, el macho es más pesado en casi 98 de cada 100 casos."

Observa las decisiones de redacción: números redondeados a la precisión que importa (800 g, no 805.0954 g), el IC traducido como "entre … y …, con 95% de confianza" y el tamaño del efecto expresado como frecuencia natural ("98 de cada 100").

| Técnico | En lenguaje llano | Qué **no** decir |
|---|---|---|
| Diferencia 805 g, IC 95% [699, 910] | "Unos 800 g más (entre 700 y 910 g, con 95% de confianza)" | "Hay 95% de probabilidad de que la diferencia real esté entre…" |
| *p* < 0.001 | "Una diferencia así sería muy improbable si los sexos no difirieran" | "Está demostrado"; "la probabilidad de error es 0.1%" |
| *p* = 0.59 | "No se encontró evidencia de diferencia; los datos son compatibles con diferencias de −86 a 150 g" | "Pesan lo mismo"; "no hay efecto" |
| d = 0.03 con *n* = 200,000 | "La diferencia es detectable, pero tan pequeña que no tiene importancia práctica" | "Diferencia altamente significativa" |
| *r* = 0.89 entre peso y consumo | "Los autos más pesados tienden a consumir más; la relación es fuerte y casi lineal" | "El peso causa el consumo"; "89% del consumo se debe al peso" |
| V de Cramér = 0.34 | "La supervivencia se asoció de forma moderada con la clase" | "La clase determinó la supervivencia" |

### 3.2 La gráfica final de un hallazgo.

Una gráfica exploratoria sirve para que **tú** descubras algo; una gráfica final sirve para que **otro** lo vea en cinco segundos. Lista de verificación:

1. **El título dice el hallazgo**, no el tipo de gráfica ("Los machos Gentoo pesan unos 800 g más", no "Masa por sexo").
2. Ejes con nombre en lenguaje llano y **unidades**.
3. Se muestran los **datos** o su dispersión, no solo una barra con la media.
4. La estimación aparece con su **incertidumbre** (IC).
5. El *n* de cada grupo está visible.
6. Nada sobra: sin cuadrículas pesadas, colores sin significado ni leyendas redundantes.

* La siguiente celda dibujará la gráfica final del hallazgo: los datos individuales, la media de cada sexo con su IC del 95% y el *n* en las etiquetas.

In [ ]:
gentoo_con_sexo = gentoo.dropna(subset=["sex"]).assign(
    sexo=lambda d: d["sex"].map({"Female": f"Hembras (n = {len(hembras)})", "Male": f"Machos (n = {len(machos)})"}))
orden = [f"Hembras (n = {len(hembras)})", f"Machos (n = {len(machos)})"]
fig, ax = plt.subplots(figsize=(7, 4))
sns.stripplot(gentoo_con_sexo, x="sexo", y="body_mass_g", order=orden, alpha=0.4, color="gray", ax=ax)
sns.pointplot(gentoo_con_sexo, x="sexo", y="body_mass_g", order=orden, errorbar=("ci", 95), color="black",
              linestyle="none", markers="D", capsize=0.1, seed=401, ax=ax)
ax.set(xlabel="", ylabel="masa corporal (g)",
       title="Los machos Gentoo pesan unos 800 g más que las hembras\n(puntos: individuos; rombos: media con IC del 95%)")
sns.despine()
plt.show()

**Interpretación:** la gráfica muestra en un solo vistazo la magnitud (la distancia entre rombos), la incertidumbre (los intervalos, muy cortos frente a la diferencia), la dispersión de los individuos y el traslape (pocas hembras superan a los machos más ligeros). **Error común que se evita:** la "gráfica de dinamita" (barra con la media y bigote de error), que esconde la distribución y el *n*.

### 3.3 Qué no afirmar.

| Tentación | Por qué no | Alternativa |
|---|---|---|
| Lenguaje causal ("provoca", "hace que", "el efecto de") con datos observacionales | No se controló la asignación; puede haber confusores | "Se asocia con", "tiende a", "en estos datos" |
| Generalizar a otra población | La muestra solo representa la población del marco | "En los modelos vendidos en EE. UU. de 1970 a 1982…" |
| "No hay diferencia" porque *p* > 0.05 | Ausencia de evidencia no es evidencia de ausencia | Reportar el IC: qué diferencias siguen siendo compatibles |
| Reportar solo lo significativo | Es *p-hacking* (sesión 6) | Reportar todas las pruebas que se hicieron |
| Precisión falsa (805.0954 g) | Sugiere una exactitud que el IC desmiente | Redondear según el ancho del IC |

📖 **Para profundizar (opcional).** *La probabilidad de superioridad en símbolos (sección 3).*

$$PS = P(X_{\text{macho}} > X_{\text{hembra}}) + \tfrac{1}{2}\,P(X_{\text{macho}} = X_{\text{hembra}}) = \frac{U}{n_{\text{machos}}\, n_{\text{hembras}}}$$

donde $U$ es el estadístico de Mann-Whitney. Si no hay diferencia, $PS = 0.5$.

**💡 Ideas clave de la sección 3:**
* Cada hallazgo responde ¿cuánto?, ¿con qué seguridad? y ¿importa?
* El tamaño del efecto se traduce a frecuencias naturales (probabilidad de superioridad).
* La gráfica final tiene título con el hallazgo, unidades, datos, IC y *n*.
* Sin lenguaje causal ni generalizaciones fuera de la población.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `26_del_notebook_al_informe.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>